# Week 9a.2: Deep Agents: The `deepagents` Library

In the previous notebook, we built a harness by hand: a to-do list, files in the graph state, and a `task` tool that runs a subagent with a fresh context. LangChain's `deepagents` library packages the same pieces, along with several more that we'll cover later in the course.

`create_deep_agent` is built on `create_agent` and its middleware, so this notebook is one of the places where we use a prebuilt agent instead of wiring the loop by hand. We'll open it up as we go, so that you can match each part of it to the code you wrote.

Documentation: https://docs.langchain.com/oss/python/deepagents/overview

The library is a separate package. Install it in your project folder with

```
uv pip install deepagents
```

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

## 0. Model Setup

Execute one of the following to define the model.

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:9b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="claude-haiku-5-5")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

## 1. The same task

We'll use the same three-part literature review and the same paper tools as in W09a.1.

In [ ]:
import time
from langchain.messages import HumanMessage

import research_tools as rt
import research_team as team

qs = list(rt.questions())
parts = [qs[3], qs[2], qs[4]]
QUESTION = "Write a short literature review for a newcomer on three questions. " + " ".join(parts)
GOLD = sorted(set(g for q in parts for g in rt.questions()[q]["gold"]))

## 2. A deep agent

`create_deep_agent` takes a model, a system prompt, and a list of subagents. 

Each subagent is a dictionary with a name, a description, a prompt, and its tools.

- The **description** is what the lead reads when it decides whom to delegate to, so it is written for the lead, the same way a tool's docstring is written for the model.
- The **prompt** is the subagent's system prompt. It is the same researcher prompt as in W09a.1.

The to-do list is not on by default in the current version of the library. `TodoListMiddleware` adds the `write_todos` tool.

In [ ]:
from deepagents import create_deep_agent
from langchain.agents.middleware import TodoListMiddleware
from langgraph.checkpoint.memory import InMemorySaver

RESEARCHER_PROMPT = """You are a research assistant. You get one piece of research to do, and the
instructions are all you know about the larger task.
Search with search_papers, read the two to four papers that matter with get_paper, and save your notes
with write_file at the path the instructions give, with every claim followed by the paper id in brackets
like [2503.13657].
Then reply with a summary of at most 80 words that names the file and the paper ids you cited.
Cite only papers you have read."""

LEAD_PROMPT = """You are the lead of a research team. Start with a to-do list. Delegate the literature
searches to the researcher subagent with the task tool, one call per part of the question, and give each
its own notes file under /notes/.
Read the notes you need, then write the report to /report.md: what is known, organized by the parts of
the question, every claim followed by the paper id in brackets as the notes give it.
When /report.md is written, reply with one sentence saying so."""

researcher = {
    "name": "researcher",
    "description": "Searches the literature on one part of a question, saves notes to a file, "
                   "and returns a short summary with paper ids.",
    "system_prompt": RESEARCHER_PROMPT,
    "tools": [rt.search_papers, rt.get_paper],
}


In [ ]:
agent = create_deep_agent(
    #TODO
)

In [ ]:
from IPython.display import Image, display
display(Image(agent.get_graph().draw_mermaid_png()))

### What the lead can call

Let's list the tools the lead's model is given. We did not pass any tools to `create_deep_agent`, so all of these come from the library.

In [ ]:
lead_tools = agent.nodes["tools"].bound.tools_by_name
list(lead_tools)

- By default the files live in the graph state, as ours did.
- `ls`, `read_file`, `write_file`, `edit_file`, `delete`, `glob`, and `grep` work on the agent's files. 
- `write_todos` is the to-do list.
- `task` creates a subagent.
- `execute` runs shell commands. Without a sandbox, it returns an error. We'll give it a sandbox on Friday.

The `task` tool's description tells the lead how to delegate:

In [ ]:
print(lead_tools["task"].description)

NOTE:

The library also adds a `general-purpose` subagent, which has the same tools as the lead. We'll ask the lead to use `researcher`, but it is allowed to pick either one.

## 3. Watching the subagents

`stream` with `subgraphs=True` reports every step, including the steps inside each subagent. Each update comes with a namespace:

- an empty namespace `()` is a step of the lead
- a namespace like `('tools:7f17f15b-...',)` is a step inside a subagent, started by a `task` call in the lead's `tools` node

The helper below prints one line per message: who wrote it and which tools it called.

In [ ]:
def who(namespace):
    return "lead" if not namespace else "subagent " + namespace[0].split(":")[1][:4]


def watch(agent, question, thread):
    config = {"configurable": {"thread_id": thread}, "recursion_limit": 80}
    start = time.time()
    for namespace, update in agent.stream({"messages": [HumanMessage(content=question)]}, config,
                                          stream_mode="updates", subgraphs=True):
        for node, values in update.items():
            messages = (values or {}).get("messages") if isinstance(values, dict) else None
            for m in (messages if isinstance(messages, list) else []):
                calls = [c["name"] for c in getattr(m, "tool_calls", None) or []]
                if m.type == "ai":
                    print(f"{time.time() - start:5.1f}s  {who(namespace):14} {', '.join(calls) or 'reply'}")
    return agent.get_state(config).values, time.time() - start


state, seconds = watch(agent, QUESTION, "review-1")

Note the lead spawned three subagents (three instances of task).

Each one searched, read papers, and saved a file on its own. The lead's next line appears only after all three have replied.

## 4. The result

The final state has the same fields as our `HarnessState`: `messages`, `todos`, and `files`. One difference is that each file is a small dictionary with the text under `content`, plus an encoding and timestamps.

In [ ]:
for t in state.get("todos", []):
    print(f"[{t['status']:11}] {t['content']}")
print()
for path, f in state["files"].items():
    print(f"{path:45} {len(f['content']):6,} characters")

In [ ]:
from IPython.display import Markdown, display

report = state["files"].get("/report.md", {}).get("content", "")
display(Markdown(report or "No report was written."))

In [ ]:
def input_tokens(messages):
    return [m.usage_metadata["input_tokens"] for m in messages if getattr(m, "usage_metadata", None)]

print("lead tool calls:")
for name, arg in team.tool_calls_of(state["messages"]):
    print(f"  {name:12} {arg}")
print(f"\ninput tokens per lead call: {input_tokens(state['messages'])}")
print(f"papers cited: {len(rt.found_ids(report))}, coverage: {rt.score_coverage(report, GOLD):.2f}, {seconds:.0f} s")

Look at the input tokens of the lead's first call. Our lead in W09a.1 started at a few hundred tokens. The library's lead starts at a few thousand, before it has done anything, because the library adds its own instructions for the to-do list, the files, and delegation to the system prompt, plus the descriptions of all its tools. That cost is paid on every call. On a long task the instructions earn their cost. On a short question, the plain Week 4 loop is cheaper.

## 5. A hand-wired subagent

A subagent does not have to be a dictionary. `CompiledSubAgent` takes any compiled graph whose state has `messages`, and the `task` tool runs it the same way. 

Here we can use the Week 7 worker from `team.build_agent`, the loop we wired by hand.
- the agent must be a compiled graph on which .invoke() can be called.

The worker has no file tools, so it returns its finding as its last message, and `task` passes that message back to the lead.

In [ ]:
from deepagents import CompiledSubAgent

worker = team.build_agent(model, team.WORKER_PROMPT)

agent_w07 = create_deep_agent(
    model=model,
    system_prompt="You lead a research team. Delegate each part of the question to the researcher "
                  "with the task tool, all in one message, then write the report, with every claim "
                  "followed by the paper id in brackets as the findings give it.",
    subagents=[CompiledSubAgent(name="researcher",
                                description="Researches one sub-question in the literature and "
                                            "returns a finding with paper ids.",
                                runnable=worker)],
    checkpointer=InMemorySaver(),
)

state_w07, seconds_w07 = watch(agent_w07, QUESTION, "review-2")

In [ ]:
report= Markdown(state_w07["messages"][-1].text)
display(report)

In [ ]:
from pathlib import Path

Path("report.md").write_text(state_w07["messages"][-1].text)

## 6. What we built, and what the library adds

| | Our harness (W09a.1) | `deepagents` |
|---|---|---|
| the loop | `build_loop`, the Week 4 loop on `HarnessState` | `create_agent` with middleware |
| the to-do list | `write_todos`, a tool that returns a `Command` | `TodoListMiddleware` |
| files | a `files` dictionary in the state, with a merge reducer | `FilesystemMiddleware` with `StateBackend`, the default |
| a subagent | `task`, a tool that invokes the researcher on one `HumanMessage` | `SubAgentMiddleware` and its `task` tool |
| a hand-wired subagent | — | `CompiledSubAgent(runnable=...)` |
| what comes back | the subagent's last message, plus its files | the same |
| memory across threads | — | a `StoreBackend` route for `/memories/`|

The library also includes several pieces we did not build. We'll cover some of them on Friday and the rest later in the course.

- **Summarization.** When the lead's context gets too long, `SummarizationMiddleware` replaces the oldest messages with a summary. It is on by default. (Later in the course)
- **Large tool results.** A tool result over 20,000 tokens is saved to a file, and the message keeps a pointer to it. (Later in the course)
- **Backends, permissions, and sandboxes.** Where the files live, which paths the agent may read or write, and a place where `execute` can safely run code. (Friday)
- **Memory across threads.** A `StoreBackend` route keeps files such as `/memories/AGENTS.md` visible from every thread.
- **Pausing for approval.** `interrupt_on` stops before chosen tool calls until a person approves.

## 7. Things to try

1. Remove `subagents=[researcher]` and run the agent again, so the lead has only the `general-purpose` subagent. Which tools does that subagent use? It has no paper tools, so what does it do?

2. Rewrite the researcher's `description` so that it is vague, for example "A helper." Does the lead still delegate to it, or does it switch to `general-purpose`?

3. Ask the lead for one part of the question at a time: "Delegate the first part, read its notes, and only then delegate the next." Watch the timing column. Does the to-do list get updated more often?

4. Count the lead's input tokens for a one-part question, such as `qs[0]`, with `create_deep_agent` and with `team.build_single`. For which kinds of questions is the harness worth its extra tokens?

In [ ]:
# your experiments here